In [ ]:
!apt-get install -y ffmpeg
!pip install matplotlib numpy

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import Video
import os

np.random.seed(7)

# =========================
# 1) Funkcje aktywacji
# =========================
def relu(x):
    return np.maximum(0.0, x)

def relu_grad(x):
    return (x > 0).astype(x.dtype)

def sigmoid(x):
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))

def bce_loss(y_true, y_pred, eps=1e-9):
    y_pred = np.clip(y_pred, eps, 1.0 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# =========================
# 2) Dane spiralne
# =========================
def make_spiral(n=600, noise=0.25):
    n2 = n // 2
    t = np.linspace(0, 2.5 * np.pi, n2)
    r = np.linspace(0.2, 1.0, n2)

    x1 = np.c_[r * np.cos(t), r * np.sin(t)] + noise * np.random.randn(n2, 2)
    x2 = np.c_[r * np.cos(t + np.pi), r * np.sin(t + np.pi)] + noise * np.random.randn(n2, 2)

    X = np.vstack([x1, x2])
    y = np.vstack([np.ones((n2, 1)), np.zeros((n2, 1))])

    idx = np.random.permutation(n)
    return X[idx], y[idx]

X, y = make_spiral(n=700, noise=0.20)
X = (X - X.mean(0)) / (X.std(0) + 1e-9)

# =========================
# 3) Sieć 2-16-16-16-1
# =========================
layer_sizes = [2, 16, 16, 16, 1]
L = len(layer_sizes) - 1

def init_params(sizes):
    W, b = [], []
    for i in range(len(sizes) - 1):
        fan_in = sizes[i]
        w = np.random.randn(sizes[i+1], fan_in) * np.sqrt(2.0 / fan_in)
        W.append(w)
        b.append(np.zeros((sizes[i+1], 1)))
    return W, b

W, b = init_params(layer_sizes)

# =========================
# 4) Forward / Backward
# =========================
def forward(X_batch, W, b):
    a = X_batch.T
    A = [a]
    Z = []

    for l in range(L):
        z = W[l] @ a + b[l]
        Z.append(z)
        if l < L - 1:
            a = relu(z)
        else:
            a = sigmoid(z)
        A.append(a)

    return A, Z

def backward(X_batch, y_batch, W, b, A, Z):
    n = X_batch.shape[0]
    yT = y_batch.T

    dW = [None] * L
    db = [None] * L

    dZ = A[-1] - yT
    dW[-1] = (dZ @ A[-2].T) / n
    db[-1] = np.mean(dZ, axis=1, keepdims=True)

    for l in reversed(range(L - 1)):
        dA = W[l+1].T @ dZ
        dZ = dA * relu_grad(Z[l])
        dW[l] = (dZ @ A[l].T) / n
        db[l] = np.mean(dZ, axis=1, keepdims=True)

    return dW, db

def update(W, b, dW, db, lr):
    for l in range(L):
        W[l] -= lr * dW[l]
        b[l] -= lr * db[l]
    return W, b

# =========================
# 5) Wizualizacja
# =========================
grid_n = 150
x_min, x_max = X[:, 0].min() - 0.7, X[:, 0].max() + 0.7
y_min, y_max = X[:, 1].min() - 0.7, X[:, 1].max() + 0.7
xx, yy = np.meshgrid(np.linspace(x_min, x_max, grid_n),
                     np.linspace(y_min, y_max, grid_n))
grid = np.c_[xx.ravel(), yy.ravel()]

def predict_proba(X_in):
    A, _ = forward(X_in, W, b)
    return A[-1].T

steps = 200
batch_size = 120
lr = 0.12

loss_hist = []

fig, ax = plt.subplots(figsize=(6, 6))

def animate(i):
    global W, b

    idx = np.random.choice(X.shape[0], batch_size, replace=False)
    Xb = X[idx]
    yb = y[idx]

    A, Z = forward(Xb, W, b)
    loss = bce_loss(yb, A[-1].T)

    dW, db = backward(Xb, yb, W, b, A, Z)
    W, b = update(W, b, dW, db, lr)

    loss_hist.append(loss)

    ax.clear()
    ax.set_title(f"Forward + Backprop | step {i+1} | loss {loss:.4f}")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    proba = predict_proba(grid).reshape(grid_n, grid_n)
    ax.contourf(xx, yy, proba, alpha=0.35)

    ax.scatter(X[y[:,0]==0][:,0], X[y[:,0]==0][:,1], s=12)
    ax.scatter(X[y[:,0]==1][:,0], X[y[:,0]==1][:,1], s=12)

ani = FuncAnimation(fig, animate, frames=steps, interval=60)

# =========================
# 6) Zapis do MP4
# =========================
filename = "forward_backprop.mp4"
ani.save(filename, writer="ffmpeg", fps=20, dpi=120)

plt.close()

Video(filename)